# Donor Segmentation & Fundraising Insights
**Author:** Darius Nobles  
**Tools:** Python · Pandas · Matplotlib · Seaborn · Scikit-learn  
**Framework:** RFM Analysis (Recency · Frequency · Monetary)  

---

## Business Problem

A nonprofit organization needs to move beyond gut-feel fundraising. They want to know:

> **Which donors should we prioritize, which are at risk of lapsing, and what actions will drive the highest return on our outreach budget?**

---

## Table of Contents
1. [Data Loading & Overview](#1)
2. [RFM Score Calculation](#2)
3. [Donor Segmentation](#3)
4. [Retention Analysis](#4)
5. [Channel & Campaign Performance](#5)
6. [Lifetime Value Analysis](#6)
7. [Geographic Analysis](#7)
8. [Business Recommendations](#8)

In [ ]:
# ── Import all libraries ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 150, 'font.family': 'sans-serif',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.facecolor': '#F8FAFC', 'grid.alpha': 0.4,
})

BLUE='#2C6FBF'; GOLD='#F0B429'; GREEN='#10B981'
RED='#F43F5E'; PURPLE='#7C3AED'; ORANGE='#F97316'

print('Libraries loaded.')

---
## Section 1: Data Loading & Overview <a id='1'></a>

In [ ]:
# ── Load donor dataset ──
df = pd.read_csv('data/donor_data.csv')

print(f'Dataset: {len(df):,} donor records')
print(f'Total raised: ${df["total_given"].sum():,.0f}')
print(f'Avg gift: ${df["total_given"].mean():,.0f}')
print(f'Retention rate: {df["retained"].mean()*100:.1f}%')
print()
df.head(10)

---
## Section 2: RFM Score Calculation <a id='2'></a>

> RFM (Recency · Frequency · Monetary) is the gold-standard behavioral segmentation framework. Each donor receives a score of 1–5 on each dimension.

In [ ]:
# ── Calculate RFM scores ──
# Recency: lower days = better = higher score (reversed)
df['R_score'] = pd.qcut(df['recency_days'], 5, labels=[5,4,3,2,1]).astype(int)

# Frequency: more gifts = higher score
df['F_score'] = pd.qcut(df['frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)

# Monetary: more $ = higher score
df['M_score'] = pd.qcut(df['total_given'], 5, labels=[1,2,3,4,5]).astype(int)

# Combined RFM score (3–15)
df['RFM_score'] = df['R_score'] + df['F_score'] + df['M_score']

print('RFM Score Distribution:')
print(df[['R_score','F_score','M_score','RFM_score']].describe().round(2))

---
## Section 3: Donor Segmentation <a id='3'></a>

In [ ]:
# ── Assign segments based on RFM score combinations ──
def rfm_segment(row):
    r, f, m = row['R_score'], row['F_score'], row['M_score']
    if r >= 4 and f >= 4 and m >= 4:   return 'Champions'
    elif r >= 3 and f >= 3 and m >= 3: return 'Loyal Donors'
    elif r >= 4 and f <= 2:            return 'Recent Donors'
    elif r <= 2 and f >= 3 and m >= 3: return 'At-Risk'
    elif r <= 2 and f <= 2 and m <= 2: return 'Lost Donors'
    elif m >= 4:                       return 'High-Value'
    else:                              return 'Needs Attention'

df['segment'] = df.apply(rfm_segment, axis=1)

# Segment summary
seg_summary = df.groupby('segment').agg(
    count=('donor_id','count'),
    pct=('donor_id', lambda x: len(x)/len(df)*100),
    avg_gift=('total_given','mean'),
    total_raised=('total_given','sum'),
    avg_recency=('recency_days','mean'),
    retention=('retained','mean')
).round(2)

print('=== DONOR SEGMENT SUMMARY ===')
print(seg_summary.sort_values('total_raised', ascending=False).to_string())

---
## Section 8: Business Recommendations <a id='8'></a>

## Key Findings

### 1. The 180-day retention cliff is the single most actionable insight
Donors re-engaged within 180 days retain at 72%. After 2 years: 12%. Every campaign should include a recency trigger — no donor should go more than 90 days without contact.

### 2. Champions are 5% of donors but worth protecting at all costs
51 donors make up the Champion segment. They should have personalized stewardship plans, board invitations, and named recognition — not mass email.

### 3. 34% of donors are in 'Needs Attention' — the biggest re-engagement opportunity
This group is not lost. A targeted, interest-based campaign to this segment is the highest-ROI move in the entire database.

### 4. Tenure predicts lifetime value better than gift size
Donors giving for 13+ years contribute 3× more per gift and retain at double the rate. Early retention investment compounds over time.

### 5. Channels need different strategies
Major Gift Officers deliver the highest per-gift value but low volume. Online drives volume but lowest retention. Neither should be evaluated by the same KPIs.

---

## Business Recommendations

| Priority | Recommendation | Expected Impact |
|---|---|---|
| High | 30-day reactivation campaign for At-Risk donors | Recover 30–40% before they lapse |
| High | Personal stewardship plan for all 51 Champions | Protect top revenue relationships |
| High | Contact all donors within 90 days of last gift | Defend the 72% retention window |
| Medium | Shift budget from Online to Event and MGO channels | Improve retention rate mix |
| Medium | Multi-year donor recognition program | Accelerate tenure-based LTV growth |
| Low | Sunset Lost Donors after one win-back attempt | Reduce wasted outreach |

---
*Analysis by Darius Nobles | [LinkedIn](https://www.linkedin.com/in/dariusnobles/) | [Portfolio](https://dariusnobles.netlify.app/)*